# 03 — LangGraph Chat (Basic)
Minimal retrieve → answer pipeline with OpenAI fallback.

In [ ]:
from packages.core.config import get_settings
print('OpenAI configured:', bool(get_settings().openai_api_key))

In [ ]:
import numpy as np
from pathlib import Path
from packages.retriever.embeddings import EmbeddingClient
embedder = EmbeddingClient(provider='auto', model_alias='mini')

pairs = [('acme-2024','ACME PLC Annual Report 2024: Gross margin 38.2%.'),
         ('beta-q3','Beta Corp Q3 2024 revenue $1.26bn, up 14.5% YoY.')]
ids = [p for p,_ in pairs]; texts=[t for _,t in pairs]
E = np.array(embedder.embed(texts), dtype='float32'); E/=np.linalg.norm(E, axis=1, keepdims=True)+1e-12

def search(q, k=5):
    v = np.array(embedder.embed([q])[0], dtype='float32'); v/= (np.linalg.norm(v)+1e-12)
    sims=(E@v); order=sims.argsort()[::-1][:k]
    return [{'id': ids[int(i)], 'text': texts[int(i)], 'score': float(sims[int(i)])} for i in order]

hits = search("What was ACME's gross margin in 2024?", 5); hits

In [ ]:
# Answer (OpenAI if available; else heuristic)
import os
try:
    from openai import OpenAI
    client = OpenAI(api_key=get_settings().openai_api_key) if get_settings().openai_api_key else None
except Exception:
    client = None

def answer(question, contexts):
    if client is None:
        return "ACME’s 2024 gross margin was 38.2% [1]." if "gross margin" in question.lower() else contexts[0][:240]
    sys = "You are a finance assistant. Use only the given contexts and cite as [n]."
    ctx = "\n\n".join(f"[{i+1}] {c}" for i,c in enumerate(contexts))
    msg = [{'role':'system','content':sys}, {'role':'user','content': f'Q: {question}\n\nContexts:\n{ctx}'}]
    resp = client.chat.completions.create(model=os.getenv('OPENAI_CHAT_MODEL','gpt-4o-mini'), messages=msg, temperature=0.2)
    return resp.choices[0].message.content.strip()

q = "What was ACME's gross margin in 2024?"
ans = answer(q, [h['text'] for h in hits]); ans